# Variant-to-function evidence agent: Colab runner
Run the cells **top to bottom, one at a time**. No zip needed: it finds the loose AlphaGenome 2 parquet files in your Google Drive.
Not yet verified against live services: the Ensembl lookup (section 3) and Open Targets (section 7). If either fails, the cell prints what to paste back.

## 1. Setup (installs everything; safe to re-run)

In [ ]:
import os, sys, subprocess, glob
from google.colab import drive
drive.mount('/content/drive')

def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(cmd)}\n{out[-1500:]}")
    return out

REPO_URL = 'https://github.com/Raj4884/Variant-to-function-evidence-agent.git'
if os.path.isdir('/content/repo/.git'):
    sh('git', '-C', '/content/repo', 'pull', '-q')
else:
    sh('git', 'clone', '-q', REPO_URL, '/content/repo')
sh(sys.executable, '-m', 'pip', 'install', '-q', 'duckdb', 'pandas', 'pyarrow', 'pydantic', 'httpx', 'pyyaml', 'pytest', 'anthropic')
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')   # no editable install needed
import variant_evidence.resolve, variant_evidence.convert, variant_evidence.agent
env = dict(os.environ, PYTHONPATH='/content/repo/src')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '/content/repo/tests'], capture_output=True, text=True, env=env, cwd='/content/repo')
print(r.stdout.strip().splitlines()[-1])

## 2. Find your AlphaGenome 2 files in Drive and inspect them

In [ ]:
import duckdb
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found anywhere in My Drive. Check it is uploaded to this Google account.')
    return hits[0]
EXPR = find('variant_scores_sc_gene_expression.parquet')
VARIANTS = find('variants.parquet')
print(EXPR)
con = duckdb.connect()
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{EXPR}')").fetchdf()[['column_name','column_type']].to_string())
print(con.execute(f"SELECT * FROM read_parquet('{EXPR}') LIMIT 2").fetchdf().T.to_string())
print(con.execute(f"SELECT channel_name, count(*) AS n FROM read_parquet('{EXPR}') GROUP BY 1 ORDER BY n DESC").fetchdf().to_string())

## 3. Turn rsIDs into genome coordinates
The AG2 files contain no rsIDs, only hg38 coordinates, so the rsIDs are looked up through the Ensembl API. If this cell fails, paste the error.

In [ ]:
from variant_evidence.resolve import resolve_rsids
RSIDS = ['rs2836882','rs12913832','rs6497271','rs1175550','rs6673829','rs2038740',
         'rs13066062','rs35705950','rs3820282','rs1421085','rs12740374']
PRIMARY = {str(i) for i in range(1, 23)} | {'X', 'Y'}
coords = resolve_rsids(RSIDS)
cand = {c: r for r, cs in coords.items() for c in cs if c[0] in PRIMARY}   # drop alt-contig duplicates

# keep only alleles that exist in the AG2 variant panel (so alt alleles are never mixed up)
pl = ','.join(str(p) for p in sorted({c[1] for c in cand}))
panel = con.execute(f"SELECT chromosome, position, ref, alt FROM read_parquet('{VARIANTS}') WHERE position IN ({pl})").fetchall()
in_panel = {(str(c).replace('chr', ''), int(p), r, a) for c, p, r, a in panel}
coord_to_rsid = {c: r for c, r in cand.items() if c in in_panel}
scored_rsids = set(coord_to_rsid.values())
for r in RSIDS:
    alleles = [c for c, rr in coord_to_rsid.items() if rr == r]
    print(r, alleles if alleles else 'NOT IN AG2 PANEL')
    if len(alleles) > 1: print('   WARNING: more than one alt allele in panel; check before trusting direction')

## 4. Pull just those variants out of the big file and normalise
Reads the 1.1 GB file with DuckDB (a few minutes). The printed effect summary is how we check the units: the paper's call threshold is |LFC| > 0.1.

In [ ]:
import pandas as pd
from variant_evidence.convert import normalise_ag2_expression, collapse_channels, write_normalised_csv
LFC_SCALE = 1.0      # set to math.log(2) only if the units turn out to be log2
OUT = '/content/drive/MyDrive/ag2_normalised.csv'

pos_list = ','.join(str(p) for p in sorted({c[1] for c in coord_to_rsid}))
cur = con.execute(f"SELECT * FROM read_parquet('{EXPR}') WHERE position IN ({pos_list})")
cols = [d[0] for d in cur.description]
per_channel = normalise_ag2_expression((dict(zip(cols, r)) for r in cur.fetchall()), coord_to_rsid, lfc_scale=LFC_SCALE)
n = write_normalised_csv(collapse_channels(per_channel), OUT)   # one row per cell type, median across assay channels
print(n, 'rows written to', OUT)
df = pd.read_csv(OUT)
print(df.groupby('variant').size().to_string())
print(df.lfc.describe().to_string())
print(df.n_agree.value_counts().sort_index().to_string())   # how many assay channels agree per row

## 5. Run the agent on every variant

In [ ]:
from variant_evidence.agent import run_agent
from variant_evidence.models import VariantQuery
from variant_evidence.providers.ag2_precomputed import AG2PrecomputedProvider

providers = [AG2PrecomputedProvider(OUT, scored_rsids=scored_rsids)]
reports = {}
for rsid in RSIDS:
    rep = run_agent(VariantQuery(rsid=rsid), providers)
    reports[rsid] = rep
    if not rep.hypotheses:
        print(f'{rsid}: ABSTAIN (no called gene effect)'); continue
    h = rep.hypotheses[0]
    print(f'{rsid}: {h.gene} {h.direction.value} conf={h.confidence.value} cells={h.top_cell_types[:2]}')
    for note in h.notes: print('    NOTE:', note)
print()
for rsid in ['rs1421085']:
    print(rsid, 'caveats:', reports[rsid].caveats)
print()
print('rs12740374 restricted to hepatocytes:')
rep = run_agent(VariantQuery(rsid='rs12740374'), providers, context=['hepatocyte'])
for h in rep.hypotheses[:4]:
    print(f'  {h.gene:8s} {h.direction.value:5s} score={h.score:.2f} conf={h.confidence.value}')
print(rep.hypotheses[0].notes if rep.hypotheses else rep.caveats)

## 6. Evaluation on the gold loci

In [ ]:
r = subprocess.run([sys.executable, '/content/repo/eval/run_eval.py', '--ag2', OUT], capture_output=True, text=True, env=env, cwd='/content/repo')
print(r.stdout + r.stderr)

## 7. Open Targets (optional, unverified)
Lists the real field names of the live API. Errors here are expected on the first try; paste the output and the code will be fixed.

In [ ]:
from variant_evidence.providers import open_targets as ot
for t in ['Variant', 'CredibleSet']:
    try: print(t, ot.introspect(t))
    except Exception as e: print(t, 'ERROR:', str(e)[:400])
try: print(ot.resolve_variant_id('rs12740374'))
except Exception as e: print('resolve ERROR:', str(e)[:400])

## 8. What to paste back into the chat
Output of sections 2, 3, 4 and 6 (and 7 if you ran it), plus any error text.